# Notebook 2: Online Feature Store Setup

Sets up the Postgres-backed Online Service, registers stream feature views with continuous aggregation for all 4 entity velocity groups, and benchmarks freshness and lookup latency.

---

### Architecture

```
FRAUD_TRANSACTIONS (source of truth)
      │
      └──► REST Ingest API ──► Online Feature Store (Postgres)
                                  ├── Stream FVs: CONTINUOUS aggregation (< 2s)
                                  └── Batch FV: Customer profile (daily)
```

### Feature Coverage

| Entity | Stream aggregations | Notes |
|---|---|---|
| Customer | count, sum, max, min, approx_count_distinct — 1h/6h/24h/48h/1wk | Primary card-testing signal |
| Merchant | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Merchant under attack |
| Wallet DPAN | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Compromised card |
| IP Address | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Bot farm |
| Customer Profile | Lifetime stats, account age | Batch online FV, daily refresh |

Derived ratio features (velocity ratios, concentration, deviation) are computed inline in the SPCS scoring container.

> **Preview note:** Requires `snowflake-ml-python >= 1.41`. Auth token is derived automatically from the active session when running inside Snowflake Notebooks.

In [ ]:
!pip install --upgrade snowflake-ml-python

import importlib
import importlib.metadata
importlib.invalidate_caches()

# The Online Feature Store API requires snowflake-ml-python >= 1.41.
v = importlib.metadata.version('snowflake-ml-python')
major, minor = int(v.split('.')[0]), int(v.split('.')[1])
if (major, minor) >= (1, 41):
    print(f'snowflake-ml-python {v} — OK')
else:
    raise RuntimeError(f'Requires >= 1.41. Found: {v}\nRun: pip install --upgrade snowflake-ml-python')

In [ ]:
import time, os, random, concurrent.futures
import numpy as np
from snowflake.snowpark.context import get_active_session

# Core Feature Store imports — public GA API
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, Entity, CreationMode
)
# OnlineConfig and StoreType moved to feature_view submodule in snowflake-ml-python >= 1.18
from snowflake.ml.feature_store.feature_view import OnlineConfig, StoreType

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA FEATURES').collect()
session.sql('USE WAREHOUSE FRAUD_DEMO_WH').collect()

# Session token — no PAT env var needed when running inside Snowflake Notebooks
token = session.connection.rest.token or os.environ.get('SNOWFLAKE_PAT', '')
if not token:
    raise RuntimeError('Could not obtain session token. If running locally set SNOWFLAKE_PAT.')
print(f'Auth token     : from active session ({token[:12]}...)')

fs = FeatureStore(
    session=session,
    database='FRAUD_DEMO_DEV',
    name='FEATURE_STORE',
    default_warehouse='FRAUD_DEMO_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print('Feature Store  : FRAUD_DEMO_DEV.FEATURE_STORE')

## 2.1 Online Feature Serving — How It Works Now

The old Postgres-backed Online Service (Preview API) has been replaced.

Online feature serving is now enabled **per feature view** via `OnlineConfig`. When a feature view
has `online_config=OnlineConfig(enable=True)`, Snowflake automatically provisions and manages an
**Online Feature Table** (backed by Hybrid Tables) for that view.

**Data flow:**
```
FRAUD_TRANSACTIONS table
  └──► Dynamic Table (offline feature view, refresh_freq controls staleness)
              └──► Online Feature Table (Hybrid Table, syncs within target_lag)
                          └──► fs.read_feature_view(store_type=StoreType.ONLINE)
```

**Freshness model:**
- `refresh_freq` — how often the Dynamic Table recomputes features from the source table (minimum 1 minute)
- `target_lag` — how quickly the Online Feature Table syncs from the DT (minimum 10 seconds)
- Effective lag = `refresh_freq` + `target_lag` ≈ 70 seconds at minimum settings

No separate service creation needed. No REST endpoints to manage.

In [ ]:
# Online serving is now configured per feature view via OnlineConfig.
# No separate service creation required — Snowflake provisions the Hybrid Table
# backing automatically when a feature view is registered with OnlineConfig(enable=True).
#
# Grant the privilege needed to create Online Feature Tables in this schema:
try:
    session.sql('''
        GRANT CREATE ONLINE FEATURE TABLE ON SCHEMA FRAUD_DEMO_DEV.FEATURES
        TO ROLE FRAUD_MLOPS
    ''').collect()
    print('CREATE ONLINE FEATURE TABLE privilege granted to FRAUD_MLOPS')
except Exception as e:
    print(f'Privilege note: {e}')

# ONLINE_CFG is passed to each feature view below.
# target_lag = how quickly the Online Hybrid Table syncs after a DT refresh (minimum 10s).
ONLINE_CFG = OnlineConfig(enable=True, target_lag='10 seconds')
print('OnlineConfig   : enable=True, target_lag=10 seconds')
print('refresh_freq   : 1 minute (set per feature view)')
print('Effective lag  : ~70 seconds from new row → online store')

## 2.2 Register Entities

In [ ]:
# An Entity is a named primary key definition — it tells the Feature Store which column(s)
# to use when joining feature views to incoming scoring requests.
# join_keys must match the column names in both the stream schema and the scoring payload.
# We define 4 entities (one per velocity dimension): Customer, Merchant, DPAN, IP Address.
# Card Token velocity is derived at scoring time from DPAN features, so no separate entity needed.
customer_entity = Entity(name='FRAUD_CUSTOMER',    join_keys=['CUSTOMER_ID'])
merchant_entity = Entity(name='FRAUD_MERCHANT',    join_keys=['MERCHANT_ID'])
dpan_entity     = Entity(name='FRAUD_WALLET_DPAN', join_keys=['WALLET_DPAN'])
ip_entity       = Entity(name='FRAUD_IP_ADDRESS',  join_keys=['IP_ADDRESS'])

# Register each entity in the Feature Store (idempotent — safe to re-run).
# Entities are lightweight metadata objects; they don't store any data themselves.
for entity in [customer_entity, merchant_entity, dpan_entity, ip_entity]:
    try:
        fs.register_entity(entity)
        print(f'  Registered: {entity.name}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'  Exists: {entity.name}')
        else:
            raise

## 2.3 Velocity Feature Views

Each entity gets a feature view backed by a **SQL Snowpark DataFrame** that computes rolling-window
aggregations over `FRAUD_TRANSACTIONS`. Snowflake manages a Dynamic Table to refresh these on
`refresh_freq`, and syncs them to an Online Feature Table within `target_lag`.

The SQL uses conditional aggregation (`COUNT_IF`, `CASE WHEN ... THEN ... END`) to compute
multiple time windows in a single pass — one GROUP BY query per entity.

In [ ]:
from snowflake.snowpark.types import DoubleType

# ── Customer velocity ─────────────────────────────────────────────────────────
# Counts, sums, and max amounts per customer across 5 time windows.
# CURRENT_TIMESTAMP() in the SQL is evaluated at DT refresh time, not query time.
# This is correct — the DT snapshot captures "velocity as of last refresh."
customer_velocity_df = session.sql("""
    SELECT
        CUSTOMER_ID,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP())) AS PURCHASES_NUM_L1H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -6,  CURRENT_TIMESTAMP())) AS PURCHASES_NUM_L6H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP())) AS PURCHASES_NUM_L24H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -48, CURRENT_TIMESTAMP())) AS PURCHASES_NUM_L48H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP())) AS PURCHASES_NUM_L1WK,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS PURCHASES_AMT_L1H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -6,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS PURCHASES_AMT_L6H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS PURCHASES_AMT_L24H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS PURCHASES_AMT_L1WK,
        MAX(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT END) AS PURCHASES_MAX_L1H,
        MAX(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT END) AS PURCHASES_MAX_L24H,
        MAX(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT END) AS PURCHASES_MAX_L1WK,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) AND MERCHANT_COUNTRY = 'GBR') AS PURCHASES_GBR_NUM_L1H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) AND MERCHANT_COUNTRY = 'GBR') AS PURCHASES_GBR_NUM_L24H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) AND MERCHANT_COUNTRY = 'GBR') AS PURCHASES_GBR_NUM_L1WK,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) AND MERCHANT_COUNTRY != 'GBR') AS PURCHASES_NONGBR_NUM_L1H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) AND MERCHANT_COUNTRY != 'GBR') AS PURCHASES_NONGBR_NUM_L24H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) AND MERCHANT_COUNTRY != 'GBR') AS PURCHASES_NONGBR_NUM_L1WK,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN MERCHANT_ID END)  AS DISTINCT_MERCHANTS_L24H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN MERCHANT_ID END)  AS DISTINCT_MERCHANTS_L1WK,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) THEN WALLET_DPAN END)  AS DISTINCT_DPANS_L1H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN WALLET_DPAN END)  AS DISTINCT_DPANS_L1WK
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    WHERE TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY CUSTOMER_ID
""")

customer_fv = FeatureView(
    name='FRAUD_CUSTOMER_VELOCITY',
    entities=[customer_entity],
    feature_df=customer_velocity_df,
    refresh_freq='1 minute',
    online_config=ONLINE_CFG,
    desc='Customer velocity — rolling window aggregations, DT-backed with online serving',
)
customer_fv = fs.register_feature_view(customer_fv, version='V1', overwrite=True, block=True)
print('Registered: FRAUD_CUSTOMER_VELOCITY')

## 2.4 Merchant, DPAN, and IP Velocity Feature Views

In [ ]:
# ── Merchant velocity ─────────────────────────────────────────────────────────
merchant_velocity_df = session.sql("""
    SELECT
        MERCHANT_ID,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP())) AS MERCHANT_TXN_NUM_L1H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -6,  CURRENT_TIMESTAMP())) AS MERCHANT_TXN_NUM_L6H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP())) AS MERCHANT_TXN_NUM_L24H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -48, CURRENT_TIMESTAMP())) AS MERCHANT_TXN_NUM_L48H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP())) AS MERCHANT_TXN_NUM_L1WK,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS MERCHANT_TXN_AMT_L1H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS MERCHANT_TXN_AMT_L24H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS MERCHANT_TXN_AMT_L1WK,
        MAX(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT END) AS MERCHANT_MAX_TXN_L24H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END) AS MERCHANT_UNIQUE_CUST_L1H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END) AS MERCHANT_UNIQUE_CUST_L24H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END) AS MERCHANT_UNIQUE_CUST_L1WK
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    WHERE TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY MERCHANT_ID
""")

merchant_fv = FeatureView(
    name='FRAUD_MERCHANT_VELOCITY',
    entities=[merchant_entity],
    feature_df=merchant_velocity_df,
    refresh_freq='1 minute',
    online_config=ONLINE_CFG,
    desc='Merchant velocity — DT-backed with online serving',
)
merchant_fv = fs.register_feature_view(merchant_fv, version='V1', overwrite=True, block=True)
print('Registered: FRAUD_MERCHANT_VELOCITY')

# ── Wallet DPAN velocity ───────────────────────────────────────────────────────
dpan_velocity_df = session.sql("""
    SELECT
        WALLET_DPAN,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP())) AS DPAN_TXN_NUM_L1H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -6,  CURRENT_TIMESTAMP())) AS DPAN_TXN_NUM_L6H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP())) AS DPAN_TXN_NUM_L24H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -48, CURRENT_TIMESTAMP())) AS DPAN_TXN_NUM_L48H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP())) AS DPAN_TXN_NUM_L1WK,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS DPAN_TXN_AMT_L1H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END)  AS DPAN_TXN_AMT_L1WK,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END) AS DPAN_DISTINCT_CUST_L1H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END)  AS DPAN_DISTINCT_CUST_L1WK
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    WHERE TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY WALLET_DPAN
""")

dpan_fv = FeatureView(
    name='FRAUD_DPAN_VELOCITY',
    entities=[dpan_entity],
    feature_df=dpan_velocity_df,
    refresh_freq='1 minute',
    online_config=ONLINE_CFG,
    desc='Wallet DPAN velocity — DT-backed with online serving',
)
dpan_fv = fs.register_feature_view(dpan_fv, version='V1', overwrite=True, block=True)
print('Registered: FRAUD_DPAN_VELOCITY')

# ── IP address velocity ────────────────────────────────────────────────────────
ip_velocity_df = session.sql("""
    SELECT
        IP_ADDRESS,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -1,  CURRENT_TIMESTAMP())) AS IP_TXN_NUM_L1H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -6,  CURRENT_TIMESTAMP())) AS IP_TXN_NUM_L6H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP())) AS IP_TXN_NUM_L24H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('hour', -48, CURRENT_TIMESTAMP())) AS IP_TXN_NUM_L48H,
        COUNT_IF(TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP())) AS IP_TXN_NUM_L1WK,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -24, CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS IP_TXN_AMT_L24H,
        SUM(CASE WHEN TRANSACTION_TS >= DATEADD('day',  -7,  CURRENT_TIMESTAMP()) THEN PURCHASE_AMOUNT ELSE 0 END) AS IP_TXN_AMT_L1WK,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('hour', -1, CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END) AS IP_DISTINCT_CUST_L1H,
        APPROX_COUNT_DISTINCT(CASE WHEN TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP()) THEN CUSTOMER_ID END)  AS IP_DISTINCT_CUST_L1WK
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    WHERE TRANSACTION_TS >= DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY IP_ADDRESS
""")

ip_fv = FeatureView(
    name='FRAUD_IP_VELOCITY',
    entities=[ip_entity],
    feature_df=ip_velocity_df,
    refresh_freq='1 minute',
    online_config=ONLINE_CFG,
    desc='IP address velocity — DT-backed with online serving',
)
ip_fv = fs.register_feature_view(ip_fv, version='V1', overwrite=True, block=True)
print('Registered: FRAUD_IP_VELOCITY')

def identity_transform(df: pd.DataFrame) -> pd.DataFrame:
    return df

# A single dummy row for backfill_df — StreamConfig requires at least one row to probe
# the transformation function during registration. This row is not used for actual features.
backfill_schema = StructType([
    StructField('CUSTOMER_ID', StringType()),
    StructField('MERCHANT_ID', StringType()),
    StructField('WALLET_DPAN', StringType()),
    StructField('IP_ADDRESS',  StringType()),
    StructField('AMOUNT_USD',  DoubleType()),
    StructField('IS_GBR',      DoubleType()),
    StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
])
backfill_df = session.create_dataframe(
    [['DUMMY', 'DUMMY', 'DUMMY', '0.0.0.0', 0.0, 0.0, datetime(2020, 1, 1)]],
    schema=backfill_schema,
)

# StreamConfig links this feature view to the FRAUD_TXN_EVENTS stream source.
# transformation_fn: identity — pass events through as-is (no pre-processing needed).
# backfill_df: single dummy row — required for probe validation during registration.
stream_cfg = StreamConfig(
    stream_source=txn_stream,
    transformation_fn=identity_transform,
    backfill_df=backfill_df,
)

# ── Customer velocity (primary card-testing signal) ──────────────────────────
# This feature view computes rolling-window aggregates per CUSTOMER_ID.
# refresh_freq='1 minute'     — the feature view materialises a new snapshot every minute.
# feature_granularity='1 minute' — the smallest time bucket for aggregation.
# timestamp_col='EVENT_TS'    — the column in the stream events that defines event time.
#
# Feature.count / sum / max / min — standard aggregates over AMOUNT_USD for each window.
# Feature.approx_count_distinct — HyperLogLog-based approximate distinct count.
#   Only supported on STRING columns for Postgres online store.
#
# Time window notation: '1h', '6h', '24h', '48h', '7d' — interpreted relative to EVENT_TS.
# .alias('...') — sets the output column name in the online store and training dataset.
#
# FeatureAggregationMethod.CONTINUOUS — features are updated as each event arrives (sub-second
# freshness), not on a batch schedule. This is what enables real-time fraud detection.
customer_fv = FeatureView(
    name='FRAUD_CUSTOMER_VELOCITY',
    entities=[customer_entity],
    stream_config=stream_cfg,
    timestamp_col='EVENT_TS',
    refresh_freq='1 minute',
    feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('PURCHASES_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('PURCHASES_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('PURCHASES_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('PURCHASES_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('PURCHASES_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '6h' ).alias('PURCHASES_AMT_L6H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('PURCHASES_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '48h').alias('PURCHASES_AMT_L48H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_MAX_L1H'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('PURCHASES_MAX_L24H'),
        Feature.max(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_MAX_L1WK'),
        Feature.min(   'AMOUNT_USD',   '24h').alias('PURCHASES_MIN_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '1h' ).alias('DISTINCT_MERCHANTS_L1H'),
        Feature.approx_count_distinct('MERCHANT_ID', '6h' ).alias('DISTINCT_MERCHANTS_L6H'),
        Feature.approx_count_distinct('MERCHANT_ID', '24h').alias('DISTINCT_MERCHANTS_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '7d' ).alias('DISTINCT_MERCHANTS_L1WK'),
        Feature.approx_count_distinct('WALLET_DPAN', '24h').alias('DISTINCT_DPANS_L24H'),
        Feature.approx_count_distinct('WALLET_DPAN', '7d' ).alias('DISTINCT_DPANS_L1WK'),
        # AMOUNT_USD is DoubleType — approx_count_distinct only supports StringType for Postgres.
        # Use count of distinct amounts via a different approach or drop this feature.
        Feature.count('IS_GBR', '24h').alias('PURCHASES_GBR_NUM_L24H'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Customer velocity — CONTINUOUS, sub-2s freshness',
)
# block=True — waits for registration to complete before returning (takes 30-90s per view).
fs.register_feature_view(customer_fv, version='V1', block=True)
print('  Registered: FRAUD_CUSTOMER_VELOCITY')

In [ ]:
# Merchant, DPAN, and IP velocity feature views are registered in cell 10 above.
# This cell verifies all 4 velocity views are registered and online-enabled.
print('Velocity feature views registered:')
for fv_name in ['FRAUD_CUSTOMER_VELOCITY', 'FRAUD_MERCHANT_VELOCITY',
                'FRAUD_DPAN_VELOCITY', 'FRAUD_IP_VELOCITY']:
    try:
        fv = fs.get_feature_view(fv_name, 'V1')
        online_status = 'online ✓' if fv.online_config and fv.online_config.enable else 'offline only'
        print(f'  {fv_name} V1 — {online_status}')
    except Exception as e:
        print(f'  {fv_name}: {e}')
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('MERCHANT_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('MERCHANT_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('MERCHANT_MAX_TXN_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('MERCHANT_UNIQUE_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('MERCHANT_UNIQUE_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('MERCHANT_UNIQUE_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Merchant velocity — CONTINUOUS',
)
fs.register_feature_view(merchant_fv, version='V1', block=True)
print('  Registered: FRAUD_MERCHANT_VELOCITY')

# ── Wallet DPAN velocity ──────────────────────────────────────────────────────
# DPAN = Device Primary Account Number (the tokenised card number provisioned to a wallet).
# Key signal: DPAN_DISTINCT_CUST_L24H > 1 means the same device token is being used by
# multiple customers — a strong indicator of DPAN cloning or account sharing fraud.
dpan_fv = FeatureView(
    name='FRAUD_DPAN_VELOCITY',
    entities=[dpan_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('DPAN_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('DPAN_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('DPAN_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('DPAN_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('DPAN_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('DPAN_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('DPAN_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='DPAN velocity — shared DPAN = fraud signal',
)
fs.register_feature_view(dpan_fv, version='V1', block=True)
print('  Registered: FRAUD_DPAN_VELOCITY')

# ── IP Address velocity ───────────────────────────────────────────────────────
# IP-level features detect botnet traffic and proxy/VPN fraud rings.
# Key signal: IP_DISTINCT_CUST_L1H > threshold means many different customers
# are transacting from the same IP in a short window — typical of a bot farm or NAT proxy.
ip_fv = FeatureView(
    name='FRAUD_IP_VELOCITY',
    entities=[ip_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('IP_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('IP_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('IP_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('IP_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('IP_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('IP_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('IP_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('IP_DISTINCT_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('IP_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('IP_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='IP velocity — shared IP = bot farm signal',
)
fs.register_feature_view(ip_fv, version='V1', block=True)
print('  Registered: FRAUD_IP_VELOCITY')
print('\nAll 4 entity stream feature views registered.')

## 2.5 Customer Profile Feature View (Batch, Daily Refresh)

Static lifetime features that change slowly. Served from a DT-backed batch online feature view refreshed daily.

In [ ]:
# The customer profile feature view is a BATCH feature view (not CONTINUOUS/stream-based).
# It reads from a pre-aggregated table (CUSTOMER_PROFILE) that is refreshed daily.
# Batch feature views are appropriate for slowly-changing attributes like:
#   - Lifetime transaction count and total spend
#   - Account age in days
#   - Average transaction amount (30-day rolling)
# These features don't need sub-second freshness — daily refresh is sufficient.

# CUSTOMER_PROFILE doesn't exist as a table — build it by joining DIM_CUSTOMERS
# (static attributes) with aggregates from FRAUD_TRANSACTIONS.
session.sql('''
    CREATE TABLE IF NOT EXISTS FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE AS
    SELECT
        c.CUSTOMER_ID,
        c.CUSTOMER_AGE,
        c.DAYS_SINCE_REGISTRATION,
        c.IS_HIGH_RISK,
        COALESCE(t.LIFETIME_TXN_COUNT, 0)    AS LIFETIME_TXN_COUNT,
        COALESCE(t.LIFETIME_TXN_TOTAL, 0.0)  AS LIFETIME_TXN_TOTAL,
        COALESCE(t.AVG_TXN_AMOUNT_30D, 0.0)  AS AVG_TXN_AMOUNT_30D
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c
    LEFT JOIN (
        SELECT
            CUSTOMER_ID,
            COUNT(*)                          AS LIFETIME_TXN_COUNT,
            SUM(PURCHASE_AMOUNT)              AS LIFETIME_TXN_TOTAL,
            AVG(CASE WHEN TRANSACTION_TS >= DATEADD(DAY, -30, CURRENT_TIMESTAMP())
                     THEN PURCHASE_AMOUNT END) AS AVG_TXN_AMOUNT_30D
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
        GROUP BY CUSTOMER_ID
    ) t ON c.CUSTOMER_ID = t.CUSTOMER_ID
''').collect()
print('Created: FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_src = session.table('FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_fv = FeatureView(
    name='FRAUD_CUSTOMER_PROFILE',
    entities=[customer_entity],
    feature_df=profile_src,
    refresh_freq='1 day',
    online_config=OnlineConfig(enable=True, target_lag='1 hour'),
    desc='Customer profile: lifetime stats, account age. Batch online FV, daily refresh.',
)
fs.register_feature_view(profile_fv, version='V1', block=True)
print('Registered: FRAUD_CUSTOMER_PROFILE (batch, daily refresh)')

## 2.6 Freshness Benchmark

Measures time from a new row landing in `FRAUD_TRANSACTIONS` to it being visible via
`fs.read_feature_view(store_type=StoreType.ONLINE)`.

**Expected lag:** `refresh_freq` (1 min) + `target_lag` (10 s) ≈ 70 seconds. This is the
effective freshness from source event to online feature lookup.

In [ ]:
import uuid

print('='*60)
print('FRESHNESS BENCHMARK')
print('='*60)
print('''
Measures time from a new row landing in FRAUD_TRANSACTIONS to
it being visible in the online feature store.

Pipeline: INSERT → Dynamic Table refresh (refresh_freq=1 min)
                 → Online Feature Table sync (target_lag=10 s)
                 → fs.read_feature_view(StoreType.ONLINE)

Expected effective lag: ~70 seconds.
''')

# Insert one test transaction with a unique customer ID
test_cust = f'FRESHTEST-{uuid.uuid4().hex[:8].upper()}'
test_merch = 'MERCH-000001'
test_dpan  = f'DPAN-{uuid.uuid4().hex[:8].upper()}'
test_ip    = 'IP-000001'

session.sql(f"""
    INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    (TRANSACTION_ID, TRANSACTION_TS, CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS,
     CARD_TOKEN, PURCHASE_AMOUNT, LOCAL_CURRENCY_CODE, MERCHANT_COUNTRY, MCC_CODE,
     TAP_AND_PAY_TYPE, WALLET_DEVICE_TYPE, WALLET_NAME,
     AUTHENTICATED_3DS_CHALLENGE_FLAG, IS_MERCHANT_INITIATED_PURCHASE, WAS_DECLINED, IS_FRAUD)
    VALUES (
        'FRESH-TEST-001', CURRENT_TIMESTAMP(),
        '{test_cust}', '{test_merch}', '{test_dpan}', '{test_ip}',
        'TOKEN-000001', 49.99, 'GBP', 'GBR', '5411',
        'NFC', 'IPHONE', 'APPLE_PAY',
        FALSE, FALSE, FALSE, FALSE
    )
""").collect()
t_insert = time.time()
print(f'Test transaction inserted for CUSTOMER_ID: {test_cust}')
print(f'Polling until customer appears in online store (max 3 minutes)...\n')

# Poll until the feature is visible in the online store
cust_fv = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
visible = False
while not visible:
    elapsed = time.time() - t_insert
    if elapsed > 180:
        print(f'Timed out after {elapsed:.0f}s — DT may still be initialising on first run.')
        break
    try:
        result = fs.read_feature_view(
            feature_view=cust_fv,
            keys=[[test_cust]],
            feature_names=['PURCHASES_NUM_L1H'],
            store_type=StoreType.ONLINE,
        ).collect()
        if result and result[0]['PURCHASES_NUM_L1H'] and result[0]['PURCHASES_NUM_L1H'] > 0:
            visible = True
            print(f'Feature visible after {elapsed:.0f}s')
            print(f'  PURCHASES_NUM_L1H = {result[0]["PURCHASES_NUM_L1H"]}  (expected: 1)')
        else:
            print(f'  [{elapsed:.0f}s] not yet visible — waiting 10s...')
            time.sleep(10)
    except Exception as e:
        print(f'  [{elapsed:.0f}s] query error: {e} — waiting 10s...')
        time.sleep(10)
print('''
What this measures:
  A single transaction event is ingested via the OFS HTTP endpoint.
  The event fans out to 4 independent CONTINUOUS feature view pipelines
  (Customer, Merchant, DPAN, IP). We poll each until its count feature
  increments from 0 → 1.

  Latency = time from ingest response to first query showing the update.
  "Time-to-score" = slowest of the 4 (all must be ready before scoring).
  Target: < 2 seconds.
''')

# The 4 feature views and their entity key columns
FV_ENTITIES = [
    ('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID'),
    ('FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID'),
    ('FRAUD_DPAN_VELOCITY',     'WALLET_DPAN'),
    ('FRAUD_IP_VELOCITY',       'IP_ADDRESS'),
]

# Generate unique entity keys for this test run
test_cust = f'CUST_{random.randint(500000, 599999)}'
test_merch = f'MERCH_{random.randint(500000, 599999)}'
test_dpan = f'DPAN_{random.randint(500000, 599999)}'
test_ip = f'IP_{random.randint(500000, 599999)}'
entity_keys = {
    'CUSTOMER_ID': test_cust,
    'MERCHANT_ID': test_merch,
    'WALLET_DPAN': test_dpan,
    'IP_ADDRESS':  test_ip,
}
# Each FV uses a count feature — we check if it increments from 0 to 1
FEATURE_PER_FV = {
    'FRAUD_CUSTOMER_VELOCITY': 'PURCHASES_NUM_L1H',
    'FRAUD_MERCHANT_VELOCITY': 'MERCHANT_TXN_NUM_L1H',
    'FRAUD_DPAN_VELOCITY':     'DPAN_TXN_NUM_L1H',
    'FRAUD_IP_VELOCITY':       'IP_TXN_NUM_L1H',
}

print(f'Test entities: {test_cust}, {test_merch}, {test_dpan}, {test_ip}')

# Step 1: Verify baselines are all 0 (fresh keys never seen before)
print('\n--- Baselines (expect all 0 for unseen keys) ---')
for fv_name, key_col in FV_ENTITIES:
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': fv_name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: entity_keys[key_col]}}],
        'features': [FEATURE_PER_FV[fv_name]],
    }, timeout=10)
    val = r.json().get('results', [{}])[0].get('features', [0])[0] or 0
    print(f'  {fv_name}: {FEATURE_PER_FV[fv_name]} = {val}')

# Step 2: Ingest a single transaction event
print('\n--- Ingest single event ---')
ingest_ts = datetime.now(timezone.utc)
ir = requests.post(f'{INGEST_URL}/api/v1/ingest', headers=HEADERS, json={
    'dry_run': False,
    'records': {'FRAUD_TXN_EVENTS': [{
        'CUSTOMER_ID': test_cust, 'MERCHANT_ID': test_merch,
        'WALLET_DPAN': test_dpan, 'IP_ADDRESS': test_ip,
        'AMOUNT_USD': round(random.uniform(10, 500), 2), 'IS_GBR': 1.0,
        'EVENT_TS': ingest_ts.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }]},
}, timeout=10)
print(f'  HTTP {ir.status_code}')
start = time.time()

# Step 3: Poll all 4 feature views until each updates
print('\n--- Polling (250ms intervals, 30s timeout) ---')
latencies = {}  # fv_name → latency_ms
pending = set(fv for fv, _ in FV_ENTITIES)

for i in range(120):  # 30s max (120 × 250ms)
    time.sleep(0.25)
    for fv_name, key_col in FV_ENTITIES:
        if fv_name not in pending:
            continue
        r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
            'name': fv_name, 'version': 'V1', 'object_type': 'feature_view',
            'request_rows': [{'entity': {key_col: entity_keys[key_col]}}],
            'features': [FEATURE_PER_FV[fv_name]],
        }, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results and (results[0].get('features', [0])[0] or 0) > 0:
                ms = (time.time() - start) * 1000
                latencies[fv_name] = ms
                pending.discard(fv_name)
                print(f'  {fv_name}: {ms:.0f}ms')
    if not pending:
        break
    if i % 8 == 0 and i > 0:
        print(f'  [{i*0.25:.1f}s] still waiting on: {", ".join(pending)}')

# Step 4: Summary
print('\n--- Results ---')
print('  Metric: end-to-end freshness (ingest response → feature queryable)')
if pending:
    print(f'  TIMEOUT: {pending} did not update within 30s')
if latencies:
    for fv, ms in sorted(latencies.items(), key=lambda x: x[1]):
        print(f'  {fv}: {ms:.0f}ms')
    print(f'\n  Fastest (single pipeline):  {min(latencies.values()):.0f}ms')
    print(f'  Slowest (time-to-score):    {max(latencies.values()):.0f}ms')
    print(f'  Verdict: {max(latencies.values()):.0f}ms {"< 2s PASS" if max(latencies.values()) < 2000 else "> 2s FAIL"}')

## 2.7 Latency Benchmark

10 warmup + 200 measured requests. Single-entity and 4-entity concurrent (per-transaction) lookups using real entity keys.

In [ ]:
print('='*60)
print('QUERY LATENCY BENCHMARK  (n=200, 4-entity concurrent)')
print('='*60)
print('''
Measures the read latency of fs.read_feature_view(StoreType.ONLINE)
for each entity. Uses real entity keys from FRAUD_TRANSACTIONS.
Fires 4 concurrent reads (customer, merchant, DPAN, IP) per request,
mirroring the production scoring pattern.
''')

# Get registered feature views
cust_fv   = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
merch_fv  = fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1')
dpan_fv   = fs.get_feature_view('FRAUD_DPAN_VELOCITY',     'V1')
ip_fv     = fs.get_feature_view('FRAUD_IP_VELOCITY',       'V1')

# Sample real entity keys
samples = session.sql('''
    SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    SAMPLE (250 ROWS)
''').collect()
print(f'Sampled {len(samples)} entity key sets\n')

def read_one(fv, keys, feature_names):
    return fs.read_feature_view(
        feature_view=fv,
        keys=[keys],
        feature_names=feature_names,
        store_type=StoreType.ONLINE,
    ).collect()

def read_4entity(row):
    t0 = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        futs = [
            pool.submit(read_one, cust_fv,  [row.CUSTOMER_ID], ['PURCHASES_NUM_L1H']),
            pool.submit(read_one, merch_fv, [row.MERCHANT_ID], ['MERCHANT_TXN_NUM_L1H']),
            pool.submit(read_one, dpan_fv,  [row.WALLET_DPAN], ['DPAN_TXN_NUM_L1H']),
            pool.submit(read_one, ip_fv,    [row.IP_ADDRESS],  ['IP_TXN_NUM_L1H']),
        ]
        [f.result() for f in concurrent.futures.as_completed(futs)]
    return (time.perf_counter() - t0) * 1000

# Warmup
print('Running 10 warmup requests...')
for _ in range(10):
    read_4entity(random.choice(samples))
print('  Warmup complete.\n')

# Measured
print('Running 200 measured requests...')
latencies = []
for i in range(200):
    latencies.append(read_4entity(random.choice(samples)))
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/200 complete...')

p = np.percentile(latencies, [50, 75, 95, 99])
print()
print('OFS Query Latency  (n=200, 4-entity concurrent, StoreType.ONLINE)')
print('─' * 60)
print(f'  p50 : {p[0]:.1f}ms')
print(f'  p75 : {p[1]:.1f}ms')
print(f'  p95 : {p[2]:.1f}ms')
print(f'  p99 : {p[3]:.1f}ms')
print(f'  max : {max(latencies):.1f}ms')
print(f'  mean: {np.mean(latencies):.1f}ms')
print('''
What this measures:
  The HTTP round-trip time to READ features from the Online Feature Store.
  No data is ingested — we are purely measuring how fast the Postgres-backed
  online store responds to point lookups by entity key.

How it differs from the Freshness Benchmark:
  • Freshness = time for a NEW event to propagate through the CONTINUOUS
    pipeline and become queryable (write path → aggregation → read path).
    It also queries all 4 feature views, but it measures the WRITE PIPELINE
    delay — how long until freshly ingested data is visible.
  • Latency = time to READ already-computed features (read path only).
    No ingestion occurs. Features already exist in the online store.
    This isolates the Postgres lookup + HTTP overhead from pipeline processing.

  In production, total scoring time = freshness + latency:
    - Freshness tells you how stale features might be at query time.
    - Latency tells you how long the scoring service blocks waiting for
      the feature vector response.

Benchmark 1: Single-entity lookup (customer only)
  → Measures raw OFS query overhead for one feature view.

Benchmark 2: 4-entity concurrent lookup (customer + merchant + DPAN + IP)
  → Simulates a real scoring call: 4 lookups fired in parallel.
  → Reports wall-clock time = max(4 individual latencies).
  → This is what the fraud model waits for per transaction.
''')

N_WARMUP, N_REQUESTS = 10, 200

# Sample 250 real entity keys from the transactions table to use as test inputs.
# Using real keys (rather than random strings) ensures cache behaviour is realistic.
samples = session.sql('SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (250 ROWS)').collect()
print(f'Sampled {len(samples)} entity keys\n')

def qlat(name, key_col, key_val):
    """Issue a single feature view query and return the round-trip latency in ms."""
    t0 = time.perf_counter()
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: key_val}}],
    }, timeout=10)
    r.raise_for_status()
    return (time.perf_counter() - t0) * 1000

# Benchmark 1: single entity
print('='*55); print('BENCHMARK 1: Single-entity (customer)'); print('='*55)
print('  Metric: HTTP round-trip for one feature view query')
lat_s = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    lat = qlat('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID'])
    if i >= N_WARMUP: lat_s.append(lat)
    if i == N_WARMUP - 1: print('  Warmup complete (first 10 requests discarded)')
arr_s = np.array(lat_s)
print(f'  n={N_REQUESTS} requests')
print(f'  p50: {np.percentile(arr_s,50):.1f}ms  p95: {np.percentile(arr_s,95):.1f}ms  p99: {np.percentile(arr_s,99):.1f}ms')

# Benchmark 2: 4-entity concurrent
print(f'\n{"="*55}'); print('BENCHMARK 2: 4-entity concurrent (per-transaction)'); print('='*55)
print('  Metric: wall-clock time for 4 parallel feature view queries')
def q_all(row):
    """Fire 4 entity lookups in parallel; return the wall-clock time (= slowest of the 4)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        fs_ = [ex.submit(qlat, 'FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID']),
               ex.submit(qlat, 'FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', row['MERCHANT_ID']),
               ex.submit(qlat, 'FRAUD_DPAN_VELOCITY',     'WALLET_DPAN',  row['WALLET_DPAN']),
               ex.submit(qlat, 'FRAUD_IP_VELOCITY',       'IP_ADDRESS',   row['IP_ADDRESS'])]
    return max(f.result() for f in fs_)

lat_m = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    wall = q_all(row)
    if i >= N_WARMUP: lat_m.append(wall)
    if i == N_WARMUP - 1: print('  Warmup complete (first 10 requests discarded)')
arr_m = np.array(lat_m)
print(f'  n={N_REQUESTS} requests')
print(f'  p50: {np.percentile(arr_m,50):.1f}ms  p95: {np.percentile(arr_m,95):.1f}ms  p99: {np.percentile(arr_m,99):.1f}ms')
print(f'\n  → This is the per-transaction feature fetch overhead in production.')

---
## Summary

| Component | Freshness | Method |
|---|---|---|
| FRAUD_CUSTOMER_VELOCITY | ~70s (refresh_freq 1 min + target_lag 10s) | DT → Online Feature Table (Hybrid Table) |
| FRAUD_MERCHANT_VELOCITY | ~70s | DT → Online Feature Table |
| FRAUD_DPAN_VELOCITY | ~70s | DT → Online Feature Table |
| FRAUD_IP_VELOCITY | ~70s | DT → Online Feature Table |
| FRAUD_CUSTOMER_PROFILE | Daily | DT → Online Feature Table |

Features are read via `fs.read_feature_view(store_type=StoreType.ONLINE)` — no REST endpoint, no separate service.

**Online Feature Table costs:** Virtual warehouse compute (key lookups + DT refresh), Hybrid Table storage (flat monthly rate/GB). Hybrid table request billing was disabled March 2026.

**Next:** NB03 — train XGBoost from the transactions table.